**Kiểm tra chất lượng dữ liệu (data quality) cho dataset Spotify songs.**

Gồm 4 phần:
1. Thiếu dữ liệu theo cột
2. Dữ liệu ngụy trang (disguised missing / giá trị vô lý / sentinel)
3. Phân bố của giá trị thiếu (MCAR / MAR / MNAR?)
4. Trùng lặp: hoàn toàn / theo khóa nghiệp vụ / tiềm ẩn

Kết quả in ra màn hình và lưu thành các file CSV trong thư mục `data/processed`.

*Thư viện*


In [1]:
import argparse
import re
import unicodedata
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats

warnings.filterwarnings("ignore")
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 70)

**Cấu hình theo file readme.md**

In [2]:
ID_COLS = ["track_id", "track_album_id", "playlist_id"]
STR_COLS = ["track_id", "track_name", "track_artist", "track_album_id",
            "track_album_name", "track_album_release_date", "playlist_name",
            "playlist_id", "playlist_genre", "playlist_subgenre"]
NUM_COLS = ["track_popularity", "danceability", "energy", "key", "loudness",
            "mode", "speechiness", "acousticness", "instrumentalness",
            "liveness", "valence", "tempo", "duration_ms"]
AUDIO_COLS = ["danceability", "energy", "key", "loudness", "mode", "speechiness",
              "acousticness", "instrumentalness", "liveness", "valence",
              "tempo", "duration_ms"]
DATE_COL = "track_album_release_date"

# Khoảng giá trị hợp lệ theo mô tả trong readme
RANGE_RULES = {
    "track_popularity": (0, 100),
    "danceability": (0, 1), "energy": (0, 1), "speechiness": (0, 1),
    "acousticness": (0, 1), "instrumentalness": (0, 1), "liveness": (0, 1),
    "valence": (0, 1),
    "key": (-1, 11),
    "mode": (0, 1),
    "loudness": (-60, 0),
}
# Các chuỗi hay dùng để "ngụy trang" giá trị thiếu
PLACEHOLDERS = {"", "na", "n/a", "nan", "null", "none", "nil", "-", "--", "---",
                "?", "??", "unknown", "undefined", "missing", "not available",
                "n.a.", "tbd", "untitled", "no data", "0", "[]", "{}"}
SENTINELS = [-1, -9, -99, -999, -9999, 999, 9999, 0]

**Chuẩn hóa chuỗi**

In [3]:

def section(title):
    print("\n" + "=" * 100)
    print(title)
    print("=" * 100)


def load(path):
    """Chỉ coi ô rỗng là NaN; giữ nguyên 'NA', 'null', 'unknown'... để còn phát hiện."""
    df = pd.read_csv(path, keep_default_na=False, na_values=[""], low_memory=False)
    df.columns = df.columns.str.strip()
    non_numeric = {}
    for c in NUM_COLS:
        if c in df and not pd.api.types.is_numeric_dtype(df[c]):
            conv = pd.to_numeric(df[c], errors="coerce")
            bad = df[c].notna() & conv.isna()
            if bad.any():
                non_numeric[c] = df.loc[bad, c].value_counts().head(10).to_dict()
            df[c] = conv
    return df, non_numeric


def parse_dates(s):
    s = s.astype("string").str.strip()
    try:
        return pd.to_datetime(s, errors="coerce", format="mixed")
    except (TypeError, ValueError):
        return pd.to_datetime(s, errors="coerce")


def norm_text(s):
    """Chuẩn hóa tên bài/nghệ sĩ để bắt trùng lặp tiềm ẩn."""
    if pd.isna(s):
        return ""
    s = unicodedata.normalize("NFKD", str(s).lower())
    s = "".join(ch for ch in s if not unicodedata.combining(ch))
    s = re.sub(r"[\(\[].*?[\)\]]", " ", s)            # bỏ (...) và [...]
    s = re.sub(r"\s-\s.*$", " ", s)                    # bỏ " - Remastered 2011"
    s = re.sub(r"\b(feat|ft|featuring)\b\.?\s.*$", " ", s)
    s = re.sub(r"[^\w\s]", " ", s)
    return re.sub(r"\s+", " ", s).strip()

**KIỂM TRA**

*1. Kiểm tra thiếu dữ liệu*

In [4]:
# ----------------------------------------------------------------------------
# 1. Thiếu dữ liệu theo cột
# ----------------------------------------------------------------------------
def check_missing(df):
    n = len(df)
    out = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "n_missing": df.isna().sum(),
    })
    out["pct_missing"] = (out["n_missing"] / n * 100).round(3)
    out["n_unique"] = df.nunique()
    out = out.sort_values("n_missing", ascending=False)

    expected = set(STR_COLS + NUM_COLS)
    absent = sorted(expected - set(df.columns))
    extra = sorted(set(df.columns) - expected)
    print(f"Số dòng: {n:,} | Số cột: {df.shape[1]}")
    print(out.to_string())
    print(f"\nTổng số dòng có ít nhất 1 ô NaN: {int(df.isna().any(axis=1).sum()):,}")
    if absent:
        print(f"[!] Cột có trong readme nhưng thiếu trong file: {absent}")
    if extra:
        print(f"[i] Cột có trong file nhưng không có trong readme: {extra}")
    return out


# ----------------------------------------------------------------------------
# 2. Dữ liệu ngụy trang
# ----------------------------------------------------------------------------
def check_disguised(df, non_numeric):
    n = len(df)
    findings = []
    mask = pd.DataFrame(False, index=df.index, columns=df.columns)

    def add(col, rule, m, to_mask=True, examples=""):
        m = pd.Series(m, index=df.index).fillna(False).astype(bool)
        k = int(m.sum())
        if k == 0:
            return
        findings.append({"column": col, "rule": rule, "n_rows": k,
                         "pct": round(k / n * 100, 3), "examples": examples})
        if to_mask and col in mask.columns:
            mask[col] |= m

    # 2.1 Chuỗi placeholder + khoảng trắng thừa
    for c in [c for c in STR_COLS if c in df.columns]:
        raw = df[c].astype("string")
        low = raw.str.strip().str.lower()
        m = low.isin(PLACEHOLDERS).fillna(False)
        ex = ", ".join(f"{k!r}:{v}" for k, v in raw[m].value_counts().head(5).items())
        add(c, "chuỗi placeholder (NA/null/unknown/-/0/rỗng...)", m, True, ex)
        add(c, "thừa khoảng trắng đầu/cuối", raw.notna() & (raw != raw.str.strip()),
            False, "")

    # 2.2 Giá trị không ép được sang số
    for c, vals in non_numeric.items():
        findings.append({"column": c, "rule": "giá trị không phải số trong cột số",
                         "n_rows": sum(vals.values()), "pct": np.nan,
                         "examples": str(vals)})

    # 2.3 Vi phạm khoảng giá trị / miền giá trị theo readme
    for c, (lo, hi) in RANGE_RULES.items():
        if c in df.columns:
            m = df[c].notna() & ((df[c] < lo) | (df[c] > hi))
            add(c, f"ngoài khoảng [{lo}, {hi}]", m, True,
                str(df.loc[m, c].value_counts().head(5).to_dict()))
    for c in ["key", "mode"]:
        if c in df.columns:
            m = df[c].notna() & (df[c] != df[c].round())
            add(c, "không phải số nguyên", m, True)
    if "mode" in df.columns:
        add("mode", "không thuộc {0,1}", df["mode"].notna() & ~df["mode"].isin([0, 1]))

    # 2.4 Giá trị "mặc định" có khả năng là thiếu
    if "key" in df.columns:
        add("key", "key = -1 (readme: không dò được khóa => thiếu)", df["key"] == -1)
    if "tempo" in df.columns:
        add("tempo", "tempo <= 0", df["tempo"] <= 0)
        add("tempo", "tempo ngoài [30, 250] BPM (đáng ngờ)",
            (df["tempo"] > 0) & ((df["tempo"] < 30) | (df["tempo"] > 250)), False)
    if "duration_ms" in df.columns:
        add("duration_ms", "duration <= 0", df["duration_ms"] <= 0)
        add("duration_ms", "duration ngoài [30 giây, 20 phút] (đáng ngờ)",
            (df["duration_ms"] > 0) &
            ((df["duration_ms"] < 30_000) | (df["duration_ms"] > 1_200_000)), False)
    if "track_popularity" in df.columns:
        add("track_popularity", "popularity = 0 (có thể là thiếu hoặc bài chưa có lượt nghe)",
            df["track_popularity"] == 0, False)
    if "loudness" in df.columns:
        add("loudness", "loudness = 0 dB đúng bằng 0 (đáng ngờ)", df["loudness"] == 0, False)
    present = [c for c in AUDIO_COLS if c in df.columns]
    if present:
        zero_row = (df[present].fillna(1) == 0).sum(axis=1) >= max(3, len(present) // 2)
        add("(nhiều cột audio)", "≥ nửa số đặc trưng audio bằng 0 trong cùng 1 dòng",
            zero_row, False)

    # 2.5 Giá trị sentinel & giá trị lặp bất thường (spike)
    for c in [c for c in NUM_COLS if c in df.columns]:
        vc = df[c].value_counts(dropna=True)
        if c in ("key", "mode"):
            continue
        top_val, top_n = vc.index[0], int(vc.iloc[0])
        if top_n / n >= 0.05:
            findings.append({"column": c, "rule": "một giá trị chiếm ≥ 5% số dòng (spike)",
                             "n_rows": top_n, "pct": round(top_n / n * 100, 3),
                             "examples": f"value={top_val}"})
        hits = {s: int(vc[s]) for s in SENTINELS if s in vc.index and s not in (0,)}
        hits = {k: v for k, v in hits.items()
                if not (c in RANGE_RULES and RANGE_RULES[c][0] <= k <= RANGE_RULES[c][1])}
        if hits:
            findings.append({"column": c, "rule": "chứa giá trị sentinel phổ biến",
                             "n_rows": sum(hits.values()),
                             "pct": round(sum(hits.values()) / n * 100, 3),
                             "examples": str(hits)})

    # 2.6 Định dạng ID (Spotify ID = 22 ký tự chữ-số)
    for c in [c for c in ID_COLS if c in df.columns]:
        s = df[c].astype("string").str.strip()
        bad = s.notna() & ~s.str.fullmatch(r"[A-Za-z0-9]{22}").fillna(False)
        add(c, "sai định dạng ID (kỳ vọng 22 ký tự chữ/số)", bad, True,
            ", ".join(map(str, s[bad].head(3))))

    # 2.7 Ngày phát hành
    if DATE_COL in df.columns:
        s = df[DATE_COL].astype("string").str.strip()
        parsed = parse_dates(s)
        nonnull = s.notna() & (s != "")
        add(DATE_COL, "không parse được thành ngày", nonnull & parsed.isna(), True,
            ", ".join(map(str, s[nonnull & parsed.isna()].head(5))))
        year_only = s.str.fullmatch(r"\d{4}").fillna(False)
        year_month = s.str.fullmatch(r"\d{4}-\d{2}").fillna(False)
        add(DATE_COL, "chỉ có năm (độ chính xác thấp)", year_only, False)
        add(DATE_COL, "chỉ có năm-tháng", year_month, False)
        add(DATE_COL, "ngày < 1900 hoặc ở tương lai",
            parsed.notna() & ((parsed < "1900-01-01") | (parsed > pd.Timestamp.today())), True,
            ", ".join(map(str, s[parsed.notna() & ((parsed < "1900-01-01") |
                                                 (parsed > pd.Timestamp.today()))].head(5))))
        add(DATE_COL, "ngày mặc định 1970-01-01 (epoch)", s == "1970-01-01", True)
        full = s.str.fullmatch(r"\d{4}-\d{2}-\d{2}").fillna(False)
        jan1 = full & s.str.endswith("-01-01").fillna(False)
        if full.sum() and jan1.sum() / full.sum() > 0.02:  # kỳ vọng ~0.27%
            findings.append({"column": DATE_COL, "rule": "ngày 01-01 chiếm tỉ lệ cao (nghi mặc định khi chỉ biết năm)",
                             "n_rows": int(jan1.sum()),
                             "pct": round(jan1.sum() / n * 100, 3),
                             "examples": f"{jan1.sum() / full.sum() * 100:.1f}% số ngày đầy đủ"})

    res = pd.DataFrame(findings)
    if res.empty:
        print("Không phát hiện dữ liệu ngụy trang theo các luật đã định nghĩa.")
    else:
        print(res.sort_values(["column", "n_rows"], ascending=[True, False]).to_string(index=False))
        print("\n(Các luật 'đáng ngờ'/'info' chỉ để rà soát thủ công, không đưa vào mask thiếu.)")
    return res, mask

*2. Phân bố của giá trị thiếu*

In [ ]:

def _longest_run(b):
    arr = b.values.astype(int)
    if arr.sum() == 0:
        return 0, 0
    edges = np.flatnonzero(np.diff(np.r_[0, arr, 0]))
    runs = edges[1::2] - edges[::2]
    return int(runs.max()), len(runs)


def _assoc_cat(miss, group, min_group=20):
    g = group.astype("object").where(group.notna(), "(NaN)")
    ct = pd.crosstab(g, miss)
    ct = ct[ct.sum(axis=1) >= min_group]
    if ct.shape[0] < 2 or ct.shape[1] < 2:
        return None
    chi2, p, _, _ = stats.chi2_contingency(ct, correction=False)
    v = np.sqrt(chi2 / (ct.values.sum() * (min(ct.shape) - 1)))
    rate = (ct[True] / ct.sum(axis=1)).sort_values(ascending=False)
    return p, v, f"nhóm cao nhất: {rate.index[0]} ({rate.iloc[0] * 100:.1f}% thiếu)"


def _assoc_num(miss, x):
    a, b = x[miss].dropna(), x[~miss].dropna()
    if len(a) < 5 or len(b) < 5:
        return None
    u, p = stats.mannwhitneyu(a, b, alternative="two-sided")
    r = abs(2 * u / (len(a) * len(b)) - 1)  # rank-biserial
    return p, r, f"median khi thiếu={a.median():.3g} vs không thiếu={b.median():.3g}"


def check_missing_pattern(df, disguised_mask):
    eff = df.isna() | disguised_mask          # "thiếu hiệu dụng" = NaN + ngụy trang
    cols = list(eff.columns[eff.any()])
    if not cols:
        print("Không có giá trị thiếu (kể cả ngụy trang) -> bỏ qua phần này.")
        return pd.DataFrame()

    n = len(df)
    print("Thiếu hiệu dụng (NaN + ngụy trang) theo cột:")
    print((eff[cols].sum().to_frame("n").assign(pct=lambda d: (d.n / n * 100).round(3))
           .sort_values("n", ascending=False)).to_string())

    # 3.1 Số ô thiếu trên mỗi dòng
    per_row = eff.sum(axis=1)
    print("\nPhân bố số cột thiếu trên mỗi dòng:")
    print(per_row.value_counts().sort_index().to_string())

    # 3.2 Đồng xuất hiện giữa các cột thiếu (phi coefficient)
    if len(cols) > 1:
        phi = eff[cols].astype(int).corr()
        pairs = [(a, b, phi.loc[a, b]) for i, a in enumerate(cols) for b in cols[i + 1:]]
        strong = [(a, b, round(r, 3)) for a, b, r in pairs if abs(r) >= 0.3]
        print("\nCặp cột thiếu cùng lúc / loại trừ nhau (|phi| >= 0.3):")
        print(pd.DataFrame(strong, columns=["col_a", "col_b", "phi"]).to_string(index=False)
              if strong else "  (không có)")

    # 3.3 Biến nhóm để kiểm tra tính hệ thống
    groups = {}
    for g in ["playlist_genre", "playlist_subgenre", "playlist_name"]:
        if g in df.columns:
            groups[g] = df[g]
    if DATE_COL in df.columns:
        year = parse_dates(df[DATE_COL]).dt.year
        groups["release_decade"] = (year // 10 * 10)
    groups["row_position_decile"] = pd.Series(pd.qcut(np.arange(n), 10, labels=False),
                                              index=df.index)

    rows = []
    for c in cols:
        miss = eff[c]
        if miss.sum() < 5:
            continue
        longest, n_runs = _longest_run(miss)
        for gname, g in groups.items():
            res = _assoc_cat(miss, g)
            if res:
                rows.append({"column": c, "vs": gname, "test": "chi2 (Cramér's V)",
                             "p_value": res[0], "effect": round(res[1], 3), "detail": res[2]})
        for f in [f for f in NUM_COLS if f in df.columns and f != c]:
            res = _assoc_num(miss, df[f])
            if res:
                rows.append({"column": c, "vs": f, "test": "Mann-Whitney (rank-biserial r)",
                             "p_value": res[0], "effect": round(res[1], 3), "detail": res[2]})
        rows.append({"column": c, "vs": "vị trí dòng (runs)", "test": "longest run",
                     "p_value": np.nan, "effect": longest,
                     "detail": f"{n_runs} đoạn liên tiếp, dài nhất {longest} dòng"})

    res_df = pd.DataFrame(rows)
    if res_df.empty:
        return res_df
    is_cat = res_df["test"].str.startswith("chi2")
    flagged = res_df[(res_df["p_value"] < 0.01) &
                     ((is_cat & (res_df["effect"] >= 0.10)) | (~is_cat & (res_df["effect"] >= 0.20)))]
    print("\nCác liên hệ đáng chú ý (p<0.01 và effect size đủ lớn; V>=0.1 hoặc |r|>=0.2):")
    print("  -> thiếu KHÔNG ngẫu nhiên hoàn toàn (MAR/MNAR). Nếu bảng rỗng: tương thích MCAR.")
    print(flagged.sort_values("effect", ascending=False).to_string(index=False)
          if len(flagged) else "  (không có)")
    runs_df = res_df[res_df["test"] == "longest run"]
    print("\nTính liên tiếp theo vị trí dòng (đoạn thiếu dài bất thường => thiếu theo lô/nguồn):")
    print(runs_df[["column", "detail"]].to_string(index=False))
    return res_df

*3. Kiểm tra dữ liệu trùng lặp*

In [6]:
def _sample_groups(df, key_cols, n_groups=5, cols=None):
    g = df[df.duplicated(key_cols, keep=False)].sort_values(key_cols)
    first = g.drop_duplicates(key_cols)[key_cols].head(n_groups)
    return g.merge(first, on=key_cols)[cols or g.columns.tolist()]


def check_duplicates(df, out_dir):
    n = len(df)
    summary = []

    # 4.1 Trùng hoàn toàn
    full_all = df.duplicated(keep=False)
    full_extra = df.duplicated(keep="first")
    summary.append({"loại": "Trùng hoàn toàn (mọi cột)", "khóa": "(all)",
                    "số dòng thuộc nhóm trùng": int(full_all.sum()),
                    "số dòng thừa (giữ 1)": int(full_extra.sum())})
    if full_all.any():
        df[full_all].sort_values(list(df.columns[:3])).to_csv(
            out_dir / "dup_full.csv", index=False, encoding="utf-8-sig")

    # 4.2 Khóa nghiệp vụ
    candidate_keys = [
        ["track_id"],
        ["track_id", "playlist_id"],
        ["track_name", "track_artist"],
        ["track_name", "track_artist", "track_album_name"],
    ]
    for key in candidate_keys:
        if not set(key) <= set(df.columns):
            continue
        d_all = df.duplicated(key, keep=False)
        summary.append({"loại": "Trùng khóa nghiệp vụ", "khóa": " + ".join(key),
                        "số dòng thuộc nhóm trùng": int(d_all.sum()),
                        "số dòng thừa (giữ 1)": int(df.duplicated(key, keep="first").sum())})
        if d_all.any():
            df[d_all].sort_values(key).to_csv(
                out_dir / f"dup_key_{'_'.join(key)}.csv", index=False, encoding="utf-8-sig")

    print(pd.DataFrame(summary).to_string(index=False))

    # Khóa chính ứng viên
    if {"track_id", "playlist_id"} <= set(df.columns):
        print("\n[i] Trong dataset này một bài hát có thể nằm ở nhiều playlist, nên "
              "track_id lặp là bình thường;\n    khóa dòng hợp lý hơn là (track_id, playlist_id).")
        k2 = df.duplicated(["track_id", "playlist_id"], keep=False)
        non_full = k2 & ~full_all
        print(f"    Dòng trùng (track_id, playlist_id) nhưng KHÁC ở cột khác: {int(non_full.sum()):,}")

    # Xung đột thuộc tính trên cùng track_id
    if "track_id" in df.columns:
        d = df[df.duplicated("track_id", keep=False)]
        if len(d):
            g = d.groupby("track_id")
            conflict = {}
            for c in df.columns:
                if c == "track_id":
                    continue
                k = int((g[c].nunique(dropna=False) > 1).sum())
                if k:
                    conflict[c] = k
            print(f"\nCùng track_id nhưng khác giá trị ở cột (số track_id bị xung đột, "
                  f"tổng {d['track_id'].nunique():,} track_id lặp):")
            if conflict:
                print(pd.Series(conflict).sort_values(ascending=False).to_string())
                print("  (playlist_* khác là hợp lệ; các cột audio/track_popularity khác là bất thường)")
            else:
                print("  (không có xung đột)")

    # 4.3 Trùng lặp tiềm ẩn (near duplicates)
    print("\n--- Trùng lặp tiềm ẩn ---")
    work = df.copy()
    work["_name_norm"] = work["track_name"].map(norm_text) if "track_name" in work else ""
    work["_artist_norm"] = work["track_artist"].map(norm_text) if "track_artist" in work else ""

    # (a) cùng tên+nghệ sĩ (thô), khác track_id
    if {"track_name", "track_artist", "track_id"} <= set(work.columns):
        valid = (work["_name_norm"] != "") & (work["_artist_norm"] != "")
        work = work[valid].reset_index(drop=True)   # bỏ dòng thiếu tên/nghệ sĩ (sẽ khớp nhầm nhau)
        g = work.groupby(["track_name", "track_artist"])["track_id"].nunique()
        raw_groups = g[g > 1]
        print(f"(a) Cùng track_name + track_artist nhưng khác track_id: {len(raw_groups):,} nhóm")

        # (b) sau chuẩn hóa (bỏ remaster/feat/dấu câu...)
        g2 = work.groupby(["_name_norm", "_artist_norm"])["track_id"].nunique()
        norm_groups = g2[g2 > 1]
        n_ids_raw = int(raw_groups.sum())
        n_ids_norm = int(norm_groups.sum())
        print(f"(b) Sau chuẩn hóa tên (bỏ (...), '- Remastered', feat., dấu câu): "
              f"{len(norm_groups):,} nhóm, {n_ids_norm:,} track_id liên quan "
              f"(trước chuẩn hóa: {n_ids_raw:,} track_id)")
        if len(norm_groups):
            keys = norm_groups.index.to_frame(index=False)
            res = work.merge(keys, on=["_name_norm", "_artist_norm"])
            res = res.sort_values(["_artist_norm", "_name_norm"])
            res.to_csv(out_dir / "dup_latent_name_artist.csv",
                                        index=False, encoding="utf-8-sig")
            show = [c for c in ["track_id", "track_name", "track_artist", "track_album_name",
                                "duration_ms", "track_popularity"] if c in res.columns]
            print(res[show].head(12).to_string(index=False))

        # (c) cùng tên chuẩn hóa + nghệ sĩ + độ dài lệch ≤ 3 giây (khả năng cao là cùng bản thu)
        if "duration_ms" in work.columns:
            w = work.drop_duplicates("track_id").sort_values(["_name_norm", "_artist_norm", "duration_ms"])
            same = (w["_name_norm"].eq(w["_name_norm"].shift()) &
                    w["_artist_norm"].eq(w["_artist_norm"].shift()) &
                    (w["duration_ms"] - w["duration_ms"].shift()).abs().le(3000) &
                    w["track_id"].ne(w["track_id"].shift()))
            pos = np.flatnonzero(same.values)
            sel = np.zeros(len(w), dtype=bool)
            sel[pos] = True
            sel[pos - 1] = True
            near = w[sel]
            print(f"(c) Cùng tên/nghệ sĩ (chuẩn hóa) và độ dài lệch ≤ 3s, khác track_id: "
                  f"{int(same.sum()):,} cặp -> khả năng cao là cùng bản thu")
            if len(near):
                near.drop(columns=["_name_norm", "_artist_norm"]).to_csv(
                    out_dir / "dup_latent_same_recording.csv", index=False, encoding="utf-8-sig")

        # (d) cùng "dấu vân tay" audio nhưng khác track_id
        feats = [c for c in AUDIO_COLS if c in work.columns]
        if feats:
            fp = work.drop_duplicates("track_id").copy()
            fp["_fp"] = pd.util.hash_pandas_object(fp[feats].round(4), index=False)
            gfp = fp.groupby("_fp")["track_id"].nunique()
            dup_fp = gfp[gfp > 1]
            print(f"(d) Cùng toàn bộ đặc trưng audio (làm tròn 4 số) nhưng khác track_id: "
                  f"{len(dup_fp):,} nhóm")
            if len(dup_fp):
                res = fp[fp["_fp"].isin(dup_fp.index)].sort_values("_fp")
                res.drop(columns=["_fp", "_name_norm", "_artist_norm"]).to_csv(
                    out_dir / "dup_latent_audio_fingerprint.csv", index=False, encoding="utf-8-sig")
                show = [c for c in ["track_id", "track_name", "track_artist", "track_album_name"]
                        if c in res.columns]
                print(res[show].head(10).to_string(index=False))

    # (e) Cùng bài xuất hiện ở nhiều playlist/genre (thông tin, không phải lỗi)
    if {"track_id", "playlist_genre"} <= set(df.columns):
        multi = df.groupby("track_id")["playlist_genre"].nunique()
        print(f"\n(e) Số track_id xuất hiện ở > 1 playlist_genre: {int((multi > 1).sum()):,} "
              f"(ảnh hưởng nhãn nếu dùng playlist_genre làm target)")

    pd.DataFrame(summary).to_csv(out_dir / "dup_summary.csv", index=False, encoding="utf-8-sig")

**Xuất dữ liệu sau tiền xử lý**

In [7]:
def export_preprocessed(df, disguised_mask, export_dir, name="spotify_songs_preprocessed.csv"):
    """Làm sạch tối thiểu (không impute, không gộp trùng tiềm ẩn) rồi lưu CSV.

- strip khoảng trắng ở các cột chuỗi
- chuyển giá trị ngụy trang (mask ở bước 2) thành NaN
- xóa các dòng trùng hoàn toàn (giữ dòng đầu)
    """
    export_dir = Path(export_dir)
    export_dir.mkdir(parents=True, exist_ok=True)

    clean = df.copy()
    for c in [c for c in STR_COLS if c in clean.columns]:
        clean[c] = clean[c].astype("string").str.strip()

    n_cells = int((disguised_mask & clean.notna()).sum().sum())
    clean = clean.mask(disguised_mask)
    for c in ["key", "mode"]:                       # giữ kiểu số nguyên sau khi có NaN
        if c in clean.columns:
            clean[c] = clean[c].round().astype("Int64")

    n_before = len(clean)
    clean = clean.drop_duplicates().reset_index(drop=True)
    path = export_dir / name
    clean.to_csv(path, index=False, encoding="utf-8")

    print(f"Đã chuyển {n_cells:,} ô ngụy trang thành NaN")
    print(f"Đã xóa {n_before - len(clean):,} dòng trùng hoàn toàn ({n_before:,} -> {len(clean):,} dòng)")
    print("Chưa xử lý: impute giá trị thiếu, trùng khóa nghiệp vụ/tiềm ẩn (cần quyết định riêng).")
    return clean


# ----------------------------------------------------------------------------
def main(argv = None):
    ap = argparse.ArgumentParser(description="Data quality checks cho Spotify songs")
    ap.add_argument(
        "csv",
        nargs="?",
        default="../data/raw/spotify_songs.csv",
        help="đường dẫn file CSV (mặc định: ../data/raw/spotify_songs.csv)",
    )
    ap.add_argument("--out", default="../data/processed/dq_report", help="thư mục lưu báo cáo kiểm tra")
    ap.add_argument(
        "--export-dir",
        default="../data/processed",
        help="thư mục xuất dữ liệu đã tiền xử lý (mặc định: ../data/processed)",
    )
    ap.add_argument("--no-export", action="store_true", help="chỉ kiểm tra, không xuất dữ liệu")

    args, _ = ap.parse_known_args(argv)
    ...

    out_dir = Path(args.out)
    out_dir.mkdir(parents=True, exist_ok=True)
    df, non_numeric = load(args.csv)

    section("1. THIẾU DỮ LIỆU THEO CỘT")
    miss = check_missing(df)
    miss.to_csv(out_dir / "1_missing_by_column.csv", encoding="utf-8-sig")

    section("2. DỮ LIỆU NGỤY TRANG")
    disguised, mask = check_disguised(df, non_numeric)
    disguised.to_csv(out_dir / "2_disguised_missing.csv", index=False, encoding="utf-8-sig")

    section("3. PHÂN BỐ CỦA GIÁ TRỊ THIẾU")
    pattern = check_missing_pattern(df, mask)
    if len(pattern):
        pattern.to_csv(out_dir / "3_missing_pattern.csv", index=False, encoding="utf-8-sig")

    section("4. TRÙNG LẶP")
    check_duplicates(df, out_dir)

    if not args.no_export:
        section("5. XUẤT DỮ LIỆU ĐÃ TIỀN XỬ LÝ")
        export_preprocessed(df, mask, args.export_dir)



if __name__ == "__main__":
    main()


1. THIẾU DỮ LIỆU THEO CỘT
Số dòng: 32,833 | Số cột: 23
                            dtype  n_missing  pct_missing  n_unique
track_id                      str          0          0.0     28356
track_name                    str          0          0.0     23450
track_artist                  str          0          0.0     10693
track_popularity            int64          0          0.0       101
track_album_id                str          0          0.0     22545
track_album_name              str          0          0.0     19744
track_album_release_date      str          0          0.0      4530
playlist_name                 str          0          0.0       449
playlist_id                   str          0          0.0       471
playlist_genre                str          0          0.0         6
playlist_subgenre             str          0          0.0        24
danceability              float64          0          0.0       822
energy                    float64          0          0.0   

In [8]:
missing = pd.read_csv('../data/processed/dq_report/1_missing_by_column.csv')
missing

,Unnamed: 0,dtype,n_missing,pct_missing,n_unique
0,track_id,str,0,0.0,28356
1,track_name,str,0,0.0,23450
2,track_artist,str,0,0.0,10693
3,track_popularity,int64,0,0.0,101
4,track_album_id,str,0,0.0,22545
5,track_album_name,str,0,0.0,19744
6,track_album_release_date,str,0,0.0,4530
7,playlist_name,str,0,0.0,449
8,playlist_id,str,0,0.0,471
9,playlist_genre,str,0,0.0,6


In [9]:
missing = pd.read_csv('../data/processed/dq_report/2_disguised_missing.csv')
missing

,column,rule,n_rows,pct,examples
0,track_name,chuỗi placeholder (NA/null/unknown/-/0/rỗng...),6,0.018,"'NA':5, '?':1"
1,track_artist,chuỗi placeholder (NA/null/unknown/-/0/rỗng...),5,0.015,'NA':5
2,track_album_name,chuỗi placeholder (NA/null/unknown/-/0/rỗng...),9,0.027,"'NA':5, '?':4"
3,loudness,"ngoài khoảng [-60, 0]",6,0.018,"{0.551: 1, 0.326: 1, 0.642: 1, 1.275: 1, 1.135: 1}"
4,tempo,tempo <= 0,1,0.003,NaN
5,duration_ms,"duration ngoài [30 giây, 20 phút] (đáng ngờ)",2,0.006,NaN
6,track_popularity,popularity = 0 (có thể là thiếu hoặc bài chưa có lượt nghe),2703,8.233,NaN
7,(nhiều cột audio),≥ nửa số đặc trưng audio bằng 0 trong cùng 1 dòng,1,0.003,NaN
8,track_popularity,một giá trị chiếm ≥ 5% số dòng (spike),2703,8.233,value=0
9,instrumentalness,một giá trị chiếm ≥ 5% số dòng (spike),12089,36.820,value=0.0


In [10]:
missing = pd.read_csv('../data/processed/dq_report/3_missing_pattern.csv')
missing

,column,vs,test,p_value,effect,detail
0,track_name,playlist_genre,chi2 (Cramér's V),0.019226,0.020,nhóm cao nhất: rap (0.1% thiếu)
1,track_name,playlist_subgenre,chi2 (Cramér's V),0.014885,0.035,nhóm cao nhất: gangster rap (0.2% thiếu)
2,track_name,playlist_name,chi2 (Cramér's V),0.030556,0.121,nhóm cao nhất: GANGSTA Rap (2.2% thiếu)
3,track_name,release_decade,chi2 (Cramér's V),0.870048,0.009,nhóm cao nhất: 2010 (0.0% thiếu)
4,track_name,row_position_decile,chi2 (Cramér's V),0.004290,0.027,nhóm cao nhất: 2 (0.1% thiếu)
...,...,...,...,...,...,...
89,loudness,liveness,Mann-Whitney (rank-biserial r),0.662916,0.103,median khi thiếu=0.141 vs không thiếu=0.127
90,loudness,valence,Mann-Whitney (rank-biserial r),0.132291,0.355,median khi thiếu=0.343 vs không thiếu=0.512
91,loudness,tempo,Mann-Whitney (rank-biserial r),0.121103,0.365,median khi thiếu=131 vs không thiếu=122
92,loudness,duration_ms,Mann-Whitney (rank-biserial r),0.115327,0.371,median khi thiếu=1.75e+05 vs không thiếu=2.16e+05


In [12]:
missing = pd.read_csv('../data/processed/dq_report/dup_summary.csv')
missing

,loại,khóa,số dòng thuộc nhóm trùng,số dòng thừa (giữ 1)
0,Trùng hoàn toàn (mọi cột),(all),0,0
1,Trùng khóa nghiệp vụ,track_id,7643,4477
2,Trùng khóa nghiệp vụ,track_id + playlist_id,1114,582
3,Trùng khóa nghiệp vụ,track_name + track_artist,10666,6603
4,Trùng khóa nghiệp vụ,track_name + track_artist + track_album_name,8829,5259


In [11]:
import pandas as pd

df = pd.read_csv("../data/processed/spotify_songs_preprocessed.csv")

# 1. Bảng tóm tắt kích thước & trùng lặp
n_before = len(df)
n_dups = int(df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)

summary_tbl = pd.DataFrame({
    "Chỉ số": [
        "Số dòng ban đầu",
        "Số cột",
        "Số dòng trùng hoàn toàn",
        "Số dòng sau khi xóa trùng",
    ],
    "Giá trị": [
        f"{n_before:,}",
        f"{df.shape[1]:,}",
        f"{n_dups:,}",
        f"{len(df):,}",
    ],
})
print("=== TÓM TẮT DỮ LIỆU ===")
print(summary_tbl.to_string(index=False))

# 2. Bảng thông tin cột: Non-Null, Missing, % Missing, Dtype
missing_cnt = df.isna().sum()
non_null_cnt = df.notna().sum()

col_tbl = pd.DataFrame({
    "#": range(len(df.columns)),
    "Column": df.columns,
    "Non-Null": non_null_cnt.values,
    "Missing": missing_cnt.values,
    "Missing (%)": (missing_cnt.values / len(df) * 100).round(2),
    "Dtype": df.dtypes.astype(str).values,
}).sort_values("Missing", ascending=False)

print("\n=== THÔNG TIN CÁC CỘT ===")
print(col_tbl.to_string(index=False))

# Nếu cần copy bảng Markdown để dán thẳng vào Docs / README:
# print("\n" + col_tbl.to_markdown(index=False))

# 3. Lưu dataset đã làm sạch
df.to_csv("../data/processed/spotify_cleaned.csv", index=False)

=== TÓM TẮT DỮ LIỆU ===
                   Chỉ số Giá trị
          Số dòng ban đầu  32,833
                   Số cột      23
  Số dòng trùng hoàn toàn       0
Số dòng sau khi xóa trùng  32,833

=== THÔNG TIN CÁC CỘT ===
 #                   Column  Non-Null  Missing  Missing (%)   Dtype
 5         track_album_name     32824        9         0.03     str
 1               track_name     32827        6         0.02     str
14                 loudness     32827        6         0.02 float64
 6 track_album_release_date     32828        5         0.02     str
 2             track_artist     32828        5         0.02     str
21                    tempo     32832        1         0.00 float64
 0                 track_id     32833        0         0.00     str
 3         track_popularity     32833        0         0.00   int64
 4           track_album_id     32833        0         0.00     str
 9           playlist_genre     32833        0         0.00     str
 7            playlist_name    